# Dunnhumby seed 43: reliability-gated N/V M2 → M5-full

- 새 학습은 **M2 1개 → M5-full 1개**만 한 GPU에서 순차 실행합니다.
- 기존 seed 43 M1과 M5-B는 완료 결과를 재사용합니다.
- 300 epoch 고정, development 684–690만 사용하며 test·holdout은 열지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess, sys, torch

SOURCE_COMMIT = '72aa4e1'
REPO = Path('/content/clv-reliability-orthogonal-nv-' + SOURCE_COMMIT)
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip() == SOURCE_COMMIT
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
sys.path.insert(0, str(REPO))
print('source:', SOURCE_COMMIT, '| GPU:', torch.cuda.get_device_name(0))


In [ ]:
import clv_reliability_orthogonal_nv_screen as screen

cfg, prepared = screen.prepare()
# 여기까지는 학습 없음. 다음 셀이 M2 후 M5-full을 순차 학습합니다.


In [ ]:
result = screen.run(cfg, prepared)
print(result['reading'])


In [ ]:
from IPython.display import display

absolute = result['absolute']
comparison = result['comparison']
display(absolute[absolute['epoch'].isin([100, 300])].sort_values(['epoch', 'model_id']))
display(comparison[comparison['epoch'].eq(300)].sort_values(['model_id', 'metric']))


In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/clv_reliability_orthogonal_nv_seed43_results', 'zip', str(Path(cfg.out_dir) / 'reports'))
files.download(archive)
